# 第 3 章：判斷 —— 讓程式做選擇

> 🎬 **情境**
> 阿宏推出會員制度，規則越加越多：
>
> 1. 會員打 9 折
> 2. 大杯加 10 元、中杯不加
> 3. 消費滿 200 再折 20
> 4. 「員工價」一律 40 元，不套用任何折扣

你昨天寫的 `單價 * 數量` 在第一條規則就死了。**價格不再是固定的，它取決於情況。**

## 3.1 😩 土法煉鋼：每種情況寫一個變數

In [ ]:
一般價 = 65
會員價 = 59
大杯加價 = 10
員工價 = 40
# ...然後呢？

卡住了。你有四個變數，但客人只會是其中一種。

你需要的不是「四個答案」，而是**「根據情況挑一個答案」的能力**。這就是 `if`。

## 3.2 🧪 先寫測試：把規則攤成表格

規則一多，用文字描述一定會漏。先寫測試，你會立刻發現自己漏想了什麼。

In [ ]:
try:
    assert 計算價格(基本價=65, 杯型="中杯", 是會員=False, 是員工=False) == 65
    assert 計算價格(基本價=65, 杯型="大杯", 是會員=False, 是員工=False) == 75
except NameError as 錯誤:
    print("🔴 紅燈：", 錯誤)

寫這幾行的過程中，你被迫問了阿宏三個他沒說清楚的問題：

| 模糊的地方 | 必須問清楚 |
| --- | --- |
| 大杯加價是在折扣**前**還是**後**？ | 先加價再打折：(65+10)×0.9 |
| 會員又是員工怎麼算？ | 員工價優先，不再打折 |
| 67.5 怎麼收？ | 四捨五入 → 68 |

> 🔍 **這是 TDD 最被低估的價值。**
> 一般人是「先寫，遇到問題再問」；TDD 是「先把問題攤開，問完再寫」。
> 後者省下的返工，遠超過寫測試花的時間。

## 3.3 💡 if / elif / else

In [ ]:
是會員 = True

if 是會員:
    價格 = 59
    print("會員價")        # 有縮排 → 屬於 if 裡面
else:
    價格 = 65
    print("原價")

print("謝謝惠顧，收您", 價格, "元")   # 沒縮排 → 一定會執行

### 三個新手必踩的點

**1️⃣ 冒號不能忘** —— 忘了會得到 `SyntaxError: expected ':'`

**2️⃣ 縮排就是範圍** —— 其他語言用 `{ }`，Python 用縮排。一層 4 個空格，不要用 Tab、不要混用。

**3️⃣ `elif` 命中就停** —— 這點最容易出錯，下面示範。

In [ ]:
分數 = 85

# ✔ 正確：elif 命中一個就不再往下
if 分數 >= 90:
    等級 = "A"
elif 分數 >= 80:
    等級 = "B"
elif 分數 >= 70:
    等級 = "C"
else:
    等級 = "D"
print("用 elif：", 等級)

In [ ]:
# ✖ 錯誤：全部用 if，每個都會檢查，後面的會覆蓋前面的
分數 = 95
if 分數 >= 90:
    等級 = "A"
if 分數 >= 80:
    等級 = "B"      # 95 分的人在這裡被改成 B
if 分數 >= 70:
    等級 = "C"      # ...然後又被改成 C
print("全用 if：", 等級, "← 95 分變成 C，錯了！")

⚠️ **順序也很重要。** 把 `>= 70` 寫在最前面，85 分會變成 C。
**寫多層 `elif` 要從最嚴格的條件開始。**

## 3.4 比較與布林運算

In [ ]:
print(65 == 65)
print(65 != 70)
print(65 >= 65)
print("珍珠" in "珍珠奶茶")
print("椰果" not in "珍珠奶茶")

In [ ]:
# Python 有個很多語言沒有的好東西：連續比較
分數 = 85
print(0 < 分數 <= 100)              # 可以這樣寫
print(0 < 分數 and 分數 <= 100)      # 其他語言要這樣寫

In [ ]:
# 邏輯運算子是英文字，不是 && || !
是會員, 金額 = True, 250
print("會員且滿 200：", 是會員 and 金額 >= 200)
print("會員或員工  ：", 是會員 or False)
print("非會員      ：", not 是會員)

### 真值判斷：什麼東西算「假」？

Python 認定為假的只有這些：`False`、`None`、`0`、`0.0`、`""`、`[]`、`{}`、`()`、`set()`

一句話：**空的、零的、None，都是假。**

In [ ]:
for 值 in [False, None, 0, 0.0, "", [], {}, (), set(), "有字", 1, [0]]:
    print(f"{repr(值):<8} → {'真' if 值 else '假'}")

In [ ]:
# ⚠️ 經典陷阱：0 是假
數量 = 0
if 數量:
    print("有輸入數量")
else:
    print("🔴 程式以為你沒輸入 —— 但其實你輸入了 0！")

# 想問「有沒有填」就要明確寫：
if 數量 is not None:
    print("🟢 正確判斷：使用者確實填了，填的是", 數量)

### ⚠️ `==` 和 `is` 的差別

In [ ]:
a = [1, 2]
b = [1, 2]
print("a == b ：", a == b, "← 內容一樣嗎")
print("a is b ：", a is b, "← 是同一個東西嗎（記憶體位置）")

c = a
print("c is a ：", c is a, "← c 和 a 是同一個東西")

**準則：比較值用 `==`；只有跟 `None` / `True` / `False` 比較時用 `is`。**

## 3.5 🟢 讓測試變綠

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

員工單價 = 40
大杯加價 = 10
會員折扣 = 0.9


def 四捨五入(數值) -> int:
    return int(Decimal(str(數值)).quantize(Decimal("1"), rounding=ROUND_HALF_UP))


def 計算價格(基本價, 杯型, 是會員, 是員工):
    # 規則 4：員工價最優先，直接回傳，後面全部不做（guard clause）
    if 是員工:
        return 員工單價

    價格 = 基本價
    if 杯型 == "大杯":
        價格 += 大杯加價

    if 是會員:
        價格 = 四捨五入(價格 * 會員折扣)

    return 價格

In [ ]:
assert 計算價格(基本價=65, 杯型="中杯", 是會員=False, 是員工=False) == 65
assert 計算價格(基本價=65, 杯型="大杯", 是會員=False, 是員工=False) == 75
assert 計算價格(基本價=65, 杯型="中杯", 是會員=True, 是員工=False) == 59
assert 計算價格(基本價=65, 杯型="大杯", 是會員=True, 是員工=False) == 68
assert 計算價格(基本價=65, 杯型="大杯", 是會員=True, 是員工=True) == 40
print("🟢 五條規則全部通過")

> 🔍 **`return` 提早結束的威力**
>
> 「員工價蓋過一切」用 `if 是員工: return 員工單價` 一行就解決了。
> 硬寫成巢狀 `if/else` 會變成縮排地獄：
>
> ```python
> if 是員工:
>     價格 = 40
> else:
>     if 杯型 == "大杯":
>         if 是會員:
>             ...          # 縮排地獄
> ```
>
> **早點 `return`，讓後面的程式不用操心已處理掉的情況** —— 這叫 *guard clause*（防衛式子句），
> 是讓程式好讀最有效的技巧之一。

## 3.6 三元運算式與 match

In [ ]:
是會員 = True
狀態 = "會員" if 是會員 else "非會員"      # 讀作：(結果A) if (條件) else (結果B)
print(狀態)

杯數, 一袋裝 = 196, 6
提袋數 = 杯數 // 一袋裝 + (1 if 杯數 % 一袋裝 else 0)
print("提袋數：", 提袋數)

In [ ]:
# match（Python 3.10+）：多個「等於某個值」的分支，比一長串 elif 清楚
def 杯型加價(杯型):
    match 杯型:
        case "大杯":
            return 10
        case "中杯":
            return 0
        case "小杯":
            return -5
        case _:                      # _ 代表「其他」
            raise ValueError(f"未知杯型：{杯型}")


print([杯型加價(x) for x in ["大杯", "中杯", "小杯"]])
assert 杯型加價("大杯") == 10
print("🟢 match 正常（需要 Python 3.10 以上）")

## 📌 本章速記

In [ ]:
# 全部用 assert 驗一遍
assert (65 if True else 70) == 65
assert (0 < 85 <= 100) is True
assert (True and False) is False
assert bool([]) is False and bool([0]) is True
assert (None is None) is True
print("🟢 速記通過")

---
## 🧪 練習

### 練習 3-1：飲料溫度建議
溫度 >= 30 回 `"冰的"`、20~29 回 `"常溫"`、< 20 回 `"熱的"`。

注意測試裡有**邊界值**（30 和 20）—— 邊界是 bug 最愛藏的地方。

In [ ]:
def 建議(溫度):
    pass  # 👉 改成你的實作


try:
    assert 建議(35) == "冰的"
    assert 建議(25) == "常溫"
    assert 建議(10) == "熱的"
    assert 建議(30) == "冰的", "邊界值 30 應該是冰的"
    assert 建議(20) == "常溫", "邊界值 20 應該是常溫"
    print("🟢 3-1 通過")
except AssertionError as 錯誤:
    print("🔴 還沒做或不正確：", 錯誤)

### 練習 3-2：滿額折扣
滿 500 折 60、滿 200 折 20、其他不折。回傳折扣後金額。

In [ ]:
def 折後金額(金額):
    pass  # 👉 改成你的實作


try:
    assert 折後金額(150) == 150
    assert 折後金額(200) == 180
    assert 折後金額(499) == 479
    assert 折後金額(500) == 440
    print("🟢 3-2 通過")
except AssertionError as 錯誤:
    print("🔴 還沒做或不正確：", 錯誤)

### 練習 3-3：找出 bug
下面這個函式在某些輸入會給錯答案。**先執行測試看它怎麼死**，再修好它。

In [ ]:
def 折後金額_有問題(金額):
    if 金額 >= 200:
        return 金額 - 20
    elif 金額 >= 500:
        return 金額 - 60
    return 金額


print("折後金額_有問題(500) =", 折後金額_有問題(500), "← 應該是 440")

try:
    assert 折後金額_有問題(500) == 440
    print("🟢 3-3 通過")
except AssertionError:
    print("🔴 抓到了！500 元的客人只折了 20。想想為什麼，然後修好上面的函式。")

---
➡️ 下一章：`04-迴圈.ipynb` —— 阿宏要你結算一整天 200 筆訂單。